In [ ]:
import os
import json
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, BertForSequenceClassification, Trainer, TrainingArguments,
    DataCollatorWithPadding
)
from transformers import TrainerCallback
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
#  BERT Fine-tune SST-2 (GLUE) — Final Grid Search Version
# ============================================================

class SST2:
    def __init__(self,
                 model_name="bert-base-uncased",
                 output_dir="./Data/BERT_SST2",
                 max_len=128,
                 batch_size=32,
                 epochs=5,
                 lr_grid=(2e-5, 3e-5),   # 🔹 2 giá trị LR mặc định
                 weight_decay=0.01,
                 seed=42):
        """Cấu hình huấn luyện BERT trên GLUE/SST-2 (tự động chạy nhiều LR)."""
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_len = max_len
        self.batch_size = batch_size
        self.epochs = epochs
        self.lr_grid = list(lr_grid)
        self.weight_decay = weight_decay
        self.seed = seed

        os.makedirs(self.output_dir, exist_ok=True)

        torch.manual_seed(seed)
        np.random.seed(seed)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        self.encoded = None
        self.raw = None
        self.model = None

    # ========================================================
    # 1️⃣ Load & preprocess data
    # ========================================================
    def load_data(self):
        self.raw = load_dataset("glue", "sst2")

        def preprocess(batch):
            return self.tokenizer(
                batch["sentence"],
                truncation=True,
                max_length=self.max_len,
                padding="max_length"
            )

        self.encoded = self.raw.map(preprocess, batched=True, remove_columns=["sentence"])
        self.encoded = self.encoded.rename_column("label", "labels")
        self.encoded.set_format(
            type="torch",
            columns=["input_ids", "token_type_ids", "attention_mask", "labels"]
        )

    # ========================================================
    # 2️⃣ Compute metrics
    # ========================================================
    @staticmethod
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
        y_pred = probs.argmax(axis=-1)
        acc = accuracy_score(labels, y_pred)
        try:
            auc = roc_auc_score(labels, probs[:, 1])
        except Exception:
            auc = float("nan")
        return {"accuracy": acc, "roc_auc": auc}

    # ========================================================
    # 3️⃣ Train model — Grid Search + ghi log ra file
    # ========================================================
    def train_model(self):
        log_path = os.path.join(self.output_dir, "train_log.txt")
        grid_path = os.path.join(self.output_dir, "grid_results.json")

        results = []
        with open(log_path, "w", encoding="utf-8") as log_f:
            for lr in self.lr_grid:
                log_f.write(f"\n# ================================ #\n")
                log_f.write(f"#         Training bert-lr{lr}      #\n")
                log_f.write(f"# ================================ #\n\n")

                model = BertForSequenceClassification.from_pretrained(
                    self.model_name, num_labels=2
                )

                args = TrainingArguments(
                    output_dir=f"{self.output_dir}/bert-lr{lr}",
                    num_train_epochs=self.epochs,
                    per_device_train_batch_size=self.batch_size,
                    per_device_eval_batch_size=self.batch_size,
                    learning_rate=lr,
                    weight_decay=self.weight_decay,
                    evaluation_strategy="epoch",
                    save_strategy="epoch",
                    load_best_model_at_end=True,
                    metric_for_best_model="accuracy",
                    greater_is_better=True,
                    fp16=torch.cuda.is_available(),
                    logging_steps=50,
                    seed=self.seed,
                    report_to="none",
                    disable_tqdm=True,
                    log_level="error"
                )

                trainer = Trainer(
                    model=model,
                    args=args,
                    train_dataset=self.encoded["train"],
                    eval_dataset=self.encoded["validation"],
                    tokenizer=self.tokenizer,
                    data_collator=self.data_collator,
                    compute_metrics=self.compute_metrics,
                )

                # ✅ Ghi log ra file bằng callback chính thống
                class FileLoggerCallback(TrainerCallback):
                    def __init__(self, file_handle):
                        self.log_f = file_handle
                    def on_log(self, args, state, control, logs=None, **kwargs):
                        if logs and "loss" in logs:
                            self.log_f.write(json.dumps(logs) + "\n")

                trainer.add_callback(FileLoggerCallback(log_f))

                trainer.train()
                eval_result = trainer.evaluate()
                log_f.write("\n" + json.dumps(eval_result, indent=2) + "\n\n")

                with open(f"{args.output_dir}/metrics_dev.json", "w", encoding="utf-8") as f:
                    json.dump({"eval": eval_result}, f, indent=2)

                results.append({
                    "lr": lr,
                    "metrics": eval_result,
                    "outdir": args.output_dir
                })

            best = max(results, key=lambda x: x["metrics"].get("eval_accuracy", float("-inf")))
            log_f.write("\nBest setting: " + json.dumps(best, indent=2) + "\n")

        with open(grid_path, "w", encoding="utf-8") as f:
            json.dump(results + [{"best": best}], f, indent=2)
        # 🔹 Ghi file tổng hợp
        with open(os.path.join(args.output_dir, "train_summary.json"), "w", encoding="utf-8") as f:
            json.dump({"results": results, "best": best}, f, indent=2)
        return results, best

    # ========================================================
    # 4️⃣ Evaluate on validation set
    # ========================================================
    def evaluate(self, final_model):
        if self.model is None:
            self.model = BertForSequenceClassification.from_pretrained(
                f"{self.output_dir}/{final_model}", num_labels=2
            )
        trainer = Trainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=self.data_collator
        )
        preds = trainer.predict(self.encoded["validation"])
        y_true = preds.label_ids
        y_prob = torch.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
        y_pred = y_prob.argmax(axis=-1)
        acc = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_prob[:, 1])

        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["neg", "pos"])
        disp.plot(cmap="Blues")
        plt.title(f"Confusion Matrix (Validation) — {final_model}")
        plt.show()

        return {"accuracy": acc, "roc_auc": auc}

    # ========================================================
    # 5️⃣ Export test predictions
    # ========================================================
    def export_test_predictions(self, final_model, checkpoint_dir=None):
        if checkpoint_dir is None:
            checkpoint_dir = f"{self.output_dir}/{final_model}"

        model = BertForSequenceClassification.from_pretrained(
            checkpoint_dir, torch_dtype=torch.float32
        ).to(self.device)

        trainer = Trainer(model=model, tokenizer=self.tokenizer, data_collator=self.data_collator)
        test_no_label = self.encoded["test"]
        if "labels" in test_no_label.column_names:
            test_no_label = test_no_label.remove_columns("labels")

        preds = trainer.predict(test_no_label)
        y_prob = torch.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
        y_pred = y_prob.argmax(axis=-1)

        idxs = self.raw["test"]["idx"]
        out_path = os.path.join(checkpoint_dir, "sst2-test-pred.tsv")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("index\tprediction\n")
            for i, p in zip(idxs, y_pred):
                f.write(f"{i}\t{int(p)}\n")


In [ ]:
# ============================================================
#  🔧 RUN CONTROL PANEL
# ============================================================

task = SST2(
    model_name="bert-base-uncased",
    output_dir="./Models/BERT_SST2",
    lr_grid=(2e-5, 3e-5),
    epochs=5,
)

task.load_data()

# results, best = task.train_model()
# final_model = os.path.relpath(best["outdir"], task.output_dir)

final_model = "final_model"
task.evaluate(final_model=final_model)
task.export_test_predictions(final_model=final_model)
